# Raining Hoops — WNBA 2026 Data Bootstrap

This notebook uses `nba_api` `LeagueGameLog` with the explicit WNBA `league_id="10"` path, plus league-level player/team season totals. It is designed for Colab and uses browser-like headers with retries.

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/samyak0407/raininghoops.git"
REPO_DIR = "/content/raininghoops"

if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

%cd /content/raininghoops
print("Repository synced:", os.getcwd())


In [ ]:
%pip install -q nba_api pandas pyarrow
print("Dependencies installed.")

In [ ]:
import time
import pandas as pd
from nba_api.stats.endpoints import leaguegamelog, leaguedashplayerstats, leaguedashteamstats

SEASON = "2026"
LEAGUE_ID = "10"  # WNBA
SEASON_TYPES = ["Regular Season", "Playoffs"]

HEADERS = {
    "Host": "stats.nba.com",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.nba.com/",
    "Origin": "https://www.nba.com",
    "Connection": "keep-alive",
}

def fetch(endpoint, retries=4, **kwargs):
    """Call an nba_api endpoint with retries and return its first DataFrame."""
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            response = endpoint(**kwargs, headers=HEADERS, timeout=60)
            frames = response.get_data_frames()
            if not frames:
                raise RuntimeError("Endpoint returned no DataFrames.")
            return frames[0]
        except Exception as exc:
            last_error = exc
            print(f"  attempt {attempt}/{retries} failed: {type(exc).__name__}: {exc}")
            if attempt < retries:
                time.sleep(5 * attempt)
    raise RuntimeError(f"Failed after {retries} attempts: {endpoint.__name__} {kwargs}") from last_error

data = {}
for season_type in SEASON_TYPES:
    tag = "reg" if season_type == "Regular Season" else "playoffs"
    print(f"\nFetching WNBA {SEASON} {season_type}...")

    # One row per team per game.
    data[f"team_games_{tag}"] = fetch(
        leaguegamelog.LeagueGameLog,
        season=SEASON,
        season_type_all_star=season_type,
        league_id=LEAGUE_ID,
        player_or_team_abbreviation="T",
        sorter="DATE",
    )
    time.sleep(1)

    # One row per player per game, including box-score production.
    data[f"player_games_{tag}"] = fetch(
        leaguegamelog.LeagueGameLog,
        season=SEASON,
        season_type_all_star=season_type,
        league_id=LEAGUE_ID,
        player_or_team_abbreviation="P",
        sorter="DATE",
    )
    time.sleep(1)

    # Season totals per player.
    data[f"player_totals_{tag}"] = fetch(
        leaguedashplayerstats.LeagueDashPlayerStats,
        season=SEASON,
        season_type_all_star=season_type,
        league_id_nullable=LEAGUE_ID,
        per_mode_detailed="Totals",
    )
    time.sleep(1)

    # Season totals per team.
    data[f"team_totals_{tag}"] = fetch(
        leaguedashteamstats.LeagueDashTeamStats,
        season=SEASON,
        season_type_all_star=season_type,
        league_id_nullable=LEAGUE_ID,
        per_mode_detailed="Totals",
    )
    time.sleep(1)

for name, df in data.items():
    path = f"{name}_{SEASON}.csv"
    df.to_csv(path, index=False)
    print(f"{name:25s} {df.shape} -> {path}")


In [ ]:
# Sanity checks before using the data downstream.
for name, df in data.items():
    print(f"\n{name}")
    print("shape:", df.shape)
    print("columns:", list(df.columns[:12]), "...")

player_reg = data["player_games_reg"]
print("\nPlayer-game sample:")
display(player_reg.head())
print("\nPlayer-game date range:", player_reg["GAME_DATE"].min(), "->", player_reg["GAME_DATE"].max())
print("Unique games:", player_reg["GAME_ID"].nunique())

if "PLAYER_ID" in player_reg.columns:
    print("Unique players:", player_reg["PLAYER_ID"].nunique())
elif "Player_ID" in player_reg.columns:
    print("Unique players:", player_reg["Player_ID"].nunique())
else:
    print("WARNING: player-game response does not expose a player ID column; inspect the columns above before downstream normalization.")


## What this gives Raining Hoops

- **Player game logs:** game-by-game production for rolling opportunity, role, and prediction features.
- **Team game logs:** game-by-game team context for pace, efficiency, opponent tendencies, and matchup features.
- **Player totals:** season-level baselines and validation features.
- **Team totals:** season-level team context.
- **Regular season + playoffs:** both are requested explicitly.

StatPick remains a separate market-intelligence source for current props, prices, and market context rather than replacing the historical game-data layer.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json

output_dir = Path("data/raw/wnba_2026")
output_dir.mkdir(parents=True, exist_ok=True)

manifest = {
    "league": "WNBA",
    "league_id": LEAGUE_ID,
    "season": SEASON,
    "season_types": SEASON_TYPES,
    "source": "WNBA Stats via nba_api LeagueGameLog / LeagueDash endpoints",
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "files": {},
}

for name, df in data.items():
    source_path = Path(f"{name}_{SEASON}.csv")
    target_path = output_dir / source_path.name
    df.to_csv(target_path, index=False)
    manifest["files"][name] = {
        "path": str(target_path),
        "rows": int(len(df)),
        "columns": int(len(df.columns)),
    }

manifest_path = output_dir / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved WNBA data and manifest to:", output_dir)
print("Manifest:", manifest_path)